# Phần 1: Bài toán Hồi quy
**Dataset:** California Housing Prices (Kaggle / StatLib)  

---

## Mục lục
- [B. EDA & Tiền xử lý Dữ liệu](#b)
  - [B.1 Load & Mô tả dataset](#b1)
  - [B.2 Xử lý Missing Values](#b2)
  - [B.3 Phân bố biến mục tiêu](#b3)
  - [B.4 Ma trận tương quan & Scatter plots](#b4)
  - [B.5 Phát hiện Outliers](#b5)
  - [B.6 Feature Engineering & Encode Categorical](#b6)
  - [B.7 Stratified Train/Val/Test Split](#b7)
  - [B.8 Chuẩn hóa (StandardScaler)](#b8)
  - [B.9 Kiểm tra phân phối sau split](#b9)
- [C. Xây dựng và huấn luyện mô hình](#c)
  - [C.1 Linear Regression](#c1)
  - [C.2 Ridge/Lasso Regression](#c2)
  - [C.3 Mô hình với Hàm Cơ sở Phi Tuyến và Ablation Study](#c3)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings, os
warnings.filterwarnings('ignore')

# Import toàn bộ helper functions từ utils.py
%load_ext autoreload
%autoreload 2
from utils import *

# Reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

DATA_PATH = '../../data/housing.csv' 
print('Import thành công!')

---
<a id='b'></a>
## B. Phân tích Khám phá & Tiền xử lý Dữ liệu

Phần này thực hiện đầy đủ pipeline EDA trước khi huấn luyện mô hình, bao gồm:
phân tích thống kê mô tả, phát hiện và xử lý missing values, phát hiện outliers,
phân tích tương quan, feature engineering, và chia dữ liệu train/val/test.

<a id='b1'></a>
### B.1 Load & Mô tả Dataset

**Dataset:** California Housing Prices, thu thập từ Census 1990 của California.  
Mỗi mẫu đại diện cho một **block group** (đơn vị địa lý nhỏ nhất của census,
thường có 600–3000 người).

| Feature | Kiểu | Mô tả |
|---|---|---|
| `longitude` | float | Kinh độ của block group |
| `latitude` | float | Vĩ độ của block group |
| `housing_median_age` | float | Tuổi trung vị của nhà trong block group |
| `total_rooms` | float | Tổng số phòng trong block group |
| `total_bedrooms` | float | Tổng số phòng ngủ (có NaN) |
| `population` | float | Dân số block group |
| `households` | float | Số hộ gia đình |
| `median_income` | float | Thu nhập trung vị (đơn vị: 10,000 USD) |
| `median_house_value` | float | **Target** — Giá nhà trung vị ($) |
| `ocean_proximity` | object | Khoảng cách tới biển (5 nhóm) |


In [ ]:
df = load_data(DATA_PATH)

In [ ]:
# Thống kê mô tả mở rộng (thêm skewness & kurtosis)
stats_df = describe_stats(df)
stats_df

**Nhận xét:**
- `total_rooms`, `total_bedrooms`, `population`, `households` có **skewness cao** (> 2),
  phân phối lệch phải mạnh — cần lưu ý khi dùng các phương pháp giả thiết Gaussian.
- `housing_median_age` bị **cap ở 52** (max = 52), tương tự `median_house_value` bị cap ở $500,000.
- `median_income` không có đơn vị USD thông thường mà đã được scale (đơn vị ≈ $10,000).

<a id='b2'></a>
### B.2 Xử lý Missing Values

Theo thống kê, chỉ có `total_bedrooms` chứa giá trị NaN.
Ta chọn **impute bằng median** vì:
1. Phân phối của `total_bedrooms` lệch phải mạnh → mean bị kéo bởi outliers.
2. Median robust hơn với outliers trong trường hợp này.

In [ ]:
# Báo cáo missing values
_ = report_missing(df)

In [ ]:
# Impute total_bedrooms bằng median
df = impute_missing(df, col='total_bedrooms', strategy='median')

# Xác nhận không còn NaN
print(f"Missing sau impute: {df.isnull().sum().sum()}")

<a id='b3'></a>
### B.3 Phân bố Biến Mục tiêu (`median_house_value`)

Trước khi xây dựng mô hình, cần hiểu rõ phân bố của biến mục tiêu.
Đây là bước quan trọng để phát hiện các vấn đề như:
- **Ceiling effect**: giá trị bị giới hạn cứng
- **Heavy-tailed distribution**: ảnh hưởng đến assumption của OLS
- **Multimodality**: có thể cần phân tách dữ liệu

In [ ]:
plot_target_distribution(df, target='median_house_value')

**Nhận xét:**
- Phân phối lệch phải, với đuôi dài về phía giá trị cao.
- Có **spike rõ rệt tại \$500,000** do **ceiling effect** — Census đặt cap tại mức này.
  Các mẫu này (~5%) không phản ánh giá trị thực → có thể ảnh hưởng đến mô hình.
- Mean (≈ \$206k) > Median (≈ \$179k), xác nhận phân phối lệch phải.
- **Hướng xử lý:** giữ lại nhưng ghi chú; có thể thử log-transform target ở phần C.

<a id='b4'></a>
### B.4 Ma trận Tương quan & Scatter Plots

Phân tích tương quan giúp:
1. Xác định features có **tương quan cao với target** → ưu tiên trong mô hình.
2. Phát hiện **multicollinearity** giữa các features → ảnh hưởng đến OLS.
3. Định hướng **feature engineering** (tạo ratio features).

In [ ]:
target_corr = plot_correlation_matrix(df, target='median_house_value')

In [ ]:
# Scatter plots các features quan trọng nhất với target
top_features = target_corr.abs().sort_values(ascending=False).head(6).index.tolist()
print(f"Top 6 features tương quan với target: {top_features}")
plot_scatter_features(df, features=top_features, target='median_house_value')

**Nhận xét:**
- `median_income` có tương quan cao nhất với target (r ≈ 0.69) — đây là predictor quan trọng nhất.
- `total_rooms`, `total_bedrooms`, `households`, `population` tương quan cao với nhau
  (multicollinearity) → Ridge/Lasso sẽ xử lý tốt hơn OLS thuần túy.
- `latitude` và `longitude` có tương quan âm — phản ánh hiệu ứng địa lý
  (vùng ven biển phía nam giá cao hơn).

<a id='b5'></a>
### B.5 Phát hiện Outliers

Hai phương pháp được sử dụng:

- **IQR method:** điểm nằm ngoài $[Q_1 - 1.5 \cdot IQR,\ Q_3 + 1.5 \cdot IQR]$  
  Phù hợp hơn cho phân phối lệch, không giả thiết Gaussian.

- **Z-score method:** điểm có $|z| > 3$ (nằm ngoài 3 độ lệch chuẩn)  
  Giả thiết phân phối Gaussian, nhạy hơn với heavy-tailed distributions.

Ta áp dụng trên các biến **tổng** (`total_rooms`, `population`, v.v.) vì chúng phụ thuộc vào
kích thước block group → phân phối lệch mạnh.

In [ ]:
outlier_report = report_outliers(df)
outlier_report

**Nhận xét:**
- `total_rooms`, `total_bedrooms`, `population`, `households` có outliers theo IQR 
  (~5–6%), phản ánh sự chênh lệch lớn về quy mô giữa các block group.
- Z-score cho kết quả ít outliers hơn vì phân phối không Gaussian (không thỏa mãn giả thiết).
- **Quyết định:** Giữ lại outliers vì đây là dữ liệu thực tế hợp lệ, không phải lỗi đo lường.
  Regularization (Ridge/Lasso) sẽ giúp giảm ảnh hưởng của chúng.

<a id='b6'></a>
### B.6 Feature Engineering & Encode Categorical

**Ratio features:** Các biến tổng (`total_rooms`, `population`) phụ thuộc vào kích thước block group,
nên ít có ý nghĩa trực tiếp. Tạo thêm ratio features có ý nghĩa thực tế hơn:

$$\text{rooms\_per\_household} = \frac{\text{total\_rooms}}{\text{households}}$$

$$\text{bedrooms\_per\_room} = \frac{\text{total\_bedrooms}}{\text{total\_rooms}}$$

$$\text{population\_per\_household} = \frac{\text{population}}{\text{households}}$$

**One-hot encoding:** `ocean_proximity` là biến categorical với 5 giá trị không có thứ tự
→ cần encode thành các biến dummy (không dùng `drop_first` để giữ khả năng giải thích).

In [ ]:
# Tạo ratio features
df = engineer_features(df)

In [ ]:
# Kiểm tra tương quan của features mới với target
new_features = ['rooms_per_household', 'bedrooms_per_room', 'population_per_household']
for f in new_features:
    from scipy.stats import pearsonr
    r, p = pearsonr(df[f], df['median_house_value'])
    print(f"  {f:<35} r = {r:+.4f}  (p = {p:.2e})")

In [ ]:
# One-hot encode ocean_proximity
df = encode_categorical(df, col='ocean_proximity')
print(f"\nShape sau encoding: {df.shape}")
print(f"Columns: {df.columns.tolist()}")

**Nhận xét:**
- `rooms_per_household` có tương quan **dương** với target (nhà nhiều phòng hơn thường đắt hơn).
- `bedrooms_per_room` có tương quan **âm** (tỷ lệ phòng ngủ cao → nhà chất lượng thấp hơn).
- `population_per_household` có tương quan **âm** (hộ đông người → khu vực thu nhập thấp hơn).
- Các ratio features này sẽ được dùng làm **basis tự chọn** trong phần E.

<a id='b7'></a>
### B.7 Stratified Train / Val / Test Split

**Tại sao stratified?**  
`median_income` là predictor quan trọng nhất và có phân phối không đều.
Nếu chia ngẫu nhiên, một tập có thể thiếu representation của nhóm thu nhập cao/thấp.
Ta chia `median_income` thành 5 nhóm và dùng `StratifiedShuffleSplit` để đảm bảo
tỷ lệ mỗi nhóm **nhất quán** trong cả 3 tập.

**Tỷ lệ:** 70% train / 10% val / 20% test

In [ ]:
X_train, X_val, X_test, y_train, y_val, y_test, feature_names = stratified_split(
    df,
    target='median_house_value',
    test_size=0.2,
    val_size=0.1,
    random_state=RANDOM_STATE
)

print(f"\nFeatures ({len(feature_names)}):")
for i, f in enumerate(feature_names):
    print(f"  [{i:>2}] {f}")

<a id='b8'></a>
### B.8 Chuẩn hóa Features (StandardScaler)

**Tại sao cần chuẩn hóa?**
- Các features có **đơn vị khác nhau**: `total_rooms` (hàng nghìn) vs `latitude` (32–38).
- OLS với Normal Equations không yêu cầu scale, nhưng **Gradient Descent** và
  **Regularization (Ridge/Lasso)** rất nhạy với scale của features.
- Ridge và Lasso phạt theo $\|\mathbf{w}\|^2$ hoặc $\|\mathbf{w}\|_1$:
  nếu không scale, features có giá trị lớn sẽ bị phạt nặng hơn một cách không công bằng.

**Quy tắc quan trọng:** Scaler chỉ được **fit trên train set**,
sau đó **transform** val và test. Không được fit trên val/test để tránh **data leakage**.

In [ ]:
X_train_s, X_val_s, X_test_s, scaler = scale_features(X_train, X_val, X_test)

print(f"\nSau scaling:")
print(f"  X_train : {X_train_s.shape}")
print(f"  X_val   : {X_val_s.shape}")
print(f"  X_test  : {X_test_s.shape}")

<a id='b9'></a>
### B.9 Kiểm tra Phân phối sau Split

Xác nhận rằng stratification đã đảm bảo phân phối target nhất quán
giữa 3 tập dữ liệu (không bị bias do quá trình chia).

In [ ]:
plot_split_distribution(y_train, y_val, y_test, target='median_house_value')

In [ ]:
# So sánh thống kê cơ bản giữa 3 tập
summary = pd.DataFrame({
    'Train': pd.Series(y_train).describe(),
    'Val'  : pd.Series(y_val).describe(),
    'Test' : pd.Series(y_test).describe(),
}).round(2)
summary

**Nhận xét:** Mean, median, std của 3 tập xấp xỉ nhau → stratification thành công.

---

## Tổng kết Phần B

| Bước | Kết quả |
|---|---|
| Missing values | 207 NaN trong `total_bedrooms` → impute bằng median |
| Outliers | Giữ lại — dữ liệu thực tế hợp lệ |
| Feature engineering | +3 ratio features, one-hot encode `ocean_proximity` (5→5 dummy cols) |
| Tổng features | 16 features sau encoding |
| Split | 14,448 train / 2,064 val / 4,128 test (stratified) |
| Scaling | StandardScaler fit trên train only |

**Dữ liệu sẵn sàng cho Phần C (Linear Regression):**
- `X_train_s`, `X_val_s`, `X_test_s` — đã scale
- `X_train`, `X_val`, `X_test` — chưa scale (dùng cho Normal Equations)
- `y_train`, `y_val`, `y_test`
- `scaler`, `feature_names`

---

<a id='c'></a>
## C. Xây dựng và Huấn luyện mô hình

<a id='c1'></a>

### C.1 Hồi quy tuyến tính (Linear Regression)

Trong phần này, chúng ta sẽ thực hiện giải bài toán Hồi quy tuyến tính bằng hai cách tiếp cận khác nhau để so sánh hiệu quả và tốc độ hội tụ:
1. **Phương trình chuẩn (Normal Equation):** Giải trực tiếp bằng đại số tuyến tính.
2. **Mini-batch Gradient Descent:** Tối ưu hóa bằng giải thuật giảm đạo hàm.


##### C.1.a Phương trình chuẩn (Normal Equation)

Phương trình chuẩn giúp tìm trực tiếp bộ tham số $\theta$ tối ưu mà không cần thực hiện các vòng lặp:
$$\hat{\theta} = (X^T X)^{-1} X^T y$$

**Lưu ý:** Với phương pháp này, chúng ta sử dụng tập dữ liệu gốc (`X_train`, `X_val`) và thêm một cột bias (giá trị bằng 1).

In [ ]:
import numpy as np
from sklearn.metrics import mean_squared_error

X_train_b = np.c_[np.ones((len(X_train), 1)), X_train].astype(float)
X_val_b = np.c_[np.ones((len(X_val), 1)), X_val].astype(float)

# Tính toán Normal Equation
X_transpose = X_train_b.T
theta_best = np.linalg.inv(X_transpose.dot(X_train_b)).dot(X_transpose).dot(y_train)

# Dự đoán trên tập Validation
y_val_predict = X_val_b.dot(theta_best)

# Tính toán sai số RMSE
rmse_normal = np.sqrt(mean_squared_error(y_val, y_val_predict))

print("--- Kết quả sau khi sửa lỗi ---")
print(f"Số lượng tham số (theta): {len(theta_best)}")
print(f"RMSE trên tập Validation: {rmse_normal:,.2f}")

**Nhận xét về Normal Equation:**
- **Ưu điểm:** Giải trực tiếp, không cần chọn Learning Rate, kết quả là tối ưu tuyệt đối trên tập huấn luyện.
- **Nhược điểm:** Chi phí tính toán ma trận nghịch đảo $O(n^3)$ sẽ rất lớn nếu số lượng đặc trưng (features) tăng cao.
- **Kết quả RMSE:** `857,529.51`

##### C.1.b Mini-batch Gradient Descent với Learning Rate Schedule

Khác với Normal Equation, Gradient Descent là một giải thuật tối ưu hóa lặp. Chúng ta sử dụng **Mini-batch** để cân bằng giữa tốc độ và sự ổn định, kết hợp với **Cosine Annealing** để điều chỉnh tốc độ học (learning rate) giảm dần theo thời gian, giúp mô hình hội tụ tốt hơn.

**Lưu ý:** Bắt buộc sử dụng dữ liệu đã chuẩn hóa (`X_train_s`, `X_val_s`) để thuật toán hội tụ.

In [ ]:
import time
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import mean_squared_error

# 1. Thêm cột bias vào dữ liệu đã scale
X_train_sm = np.c_[np.ones((len(X_train_s), 1)), X_train_s].astype(float)
X_val_sm = np.c_[np.ones((len(X_val_s), 1)), X_val_s].astype(float)

# 2. Cấu hình Hyperparameters
n_epochs = 100
batch_size = 32
eta_max = 0.0001
eta_min = 0.000001

def cosine_annealing(epoch, n_epochs, eta_max, eta_min):
    return eta_min + 0.5 * (eta_max - eta_min) * (1 + np.cos(np.pi * epoch / n_epochs))

# 3. Huấn luyện Mini-batch GD
m = len(X_train_sm)
theta_gd = np.random.randn(X_train_sm.shape[1], 1) # Khởi tạo ngẫu nhiên

y_train_rect = np.array(y_train).reshape(-1, 1)
y_val_rect = np.array(y_val).reshape(-1, 1) 
# -----------------------

loss_history = []
start_time = time.time()

for epoch in range(n_epochs):
    shuffled_indices = np.random.permutation(m)
    X_shuffled = X_train_sm[shuffled_indices]
    y_shuffled = y_train_rect[shuffled_indices]
    
    eta = cosine_annealing(epoch, n_epochs, eta_max, eta_min)
    
    for i in range(0, m, batch_size):
        xi = X_shuffled[i:i+batch_size]
        yi = y_shuffled[i:i+batch_size]
        gradients = 2/len(xi) * xi.T.dot(xi.dot(theta_gd) - yi)
        theta_gd = theta_gd - eta * gradients
    
    y_train_pred = X_train_sm.dot(theta_gd)
    loss = np.sqrt(mean_squared_error(y_train_rect, y_train_pred))
    loss_history.append(loss)

print(f"Max theta: {np.max(np.abs(theta_gd))}")

gd_time = time.time() - start_time
print(f"Thời gian huấn luyện GD: {gd_time:.4f}s")

# 4. Dự đoán và so sánh
y_val_pred_gd = X_val_sm.dot(theta_gd)
rmse_gd = np.sqrt(mean_squared_error(y_val_rect, y_val_pred_gd))
print(f"RMSE (GD) trên tập Validation: {rmse_gd:,.2f}")

# Vẽ đồ thị hội tụ
plt.figure(figsize=(8, 5))
plt.plot(loss_history, color='blue', label='Train RMSE')
plt.xlabel("Epoch")
plt.ylabel("RMSE")
plt.title("Sự hội tụ của Mini-batch Gradient Descent (Cosine Annealing)")
plt.legend()
plt.grid(True)
plt.show()

##### C.1.c Kiểm tra giả thiết Gauss-Markov


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

# Tính toán Residuals
# Đảm bảo residuals là mảng 1D để dễ xử lý
residuals = (np.array(y_val).flatten() - y_val_pred_gd.flatten())
y_pred = y_val_pred_gd.flatten()

# Khởi tạo figure
fig, ax = plt.subplots(1, 2, figsize=(15, 5))

# --- Residual Plot---
sns.scatterplot(x=y_pred, y=residuals, ax=ax[0], color='teal', alpha=0.5)
ax[0].axhline(y=0, color='r', linestyle='--')
ax[0].set_xlabel("Giá trị dự báo (Predicted)")
ax[0].set_ylabel("Phần dư (Residuals)")
ax[0].set_title("1. Residual Plot")

# --- Q-Plot ---
# Sắp xếp residuals
res_sorted = np.sort(residuals)
# Tạo các điểm phân vị lý thuyết của phân phối chuẩn
# Sử dụng phương pháp mô phỏng đơn giản từ numpy nếu không có scipy.stats.ppf
n = len(res_sorted)
theoretical_quantiles = np.random.normal(0, 1, n)
theoretical_quantiles.sort()

# Chuẩn hóa residuals để so sánh với phân phối chuẩn (0, 1)
res_standardized = (res_sorted - np.mean(res_sorted)) / np.std(res_sorted)

ax[1].scatter(theoretical_quantiles, res_standardized, color='blue', alpha=0.5)
# Vẽ đường chéo 45 độ
line = np.linspace(np.min(theoretical_quantiles), np.max(theoretical_quantiles), 100)
ax[1].plot(line, line, color='red', linestyle='--')
ax[1].set_xlabel("Theoretical Quantiles")
ax[1].set_ylabel("Sample Quantiles (Standardized)")
ax[1].set_title("2. Normal QQ-plot (Manual)")

plt.tight_layout()
plt.show()

# --- Breusch–Pagan Test ---
# Bước 1: Bình phương phần dư
residuals_sq = residuals**2

# Bước 2: Hồi quy bình phương phần dư theo các biến độc lập X
bp_model = LinearRegression()
bp_model.fit(X_val_sm, residuals_sq)
y_sq_pred = bp_model.predict(X_val_sm)

# Bước 3: Tính toán R-squared của mô hình phụ này
r2_bp = r2_score(residuals_sq, y_sq_pred)

# Bước 4: Tính LM Statistic = n * R2
n_samples = len(residuals)
lm_statistic = n_samples * r2_bp

print("--- KẾT QUẢ PHÂN TÍCH PHƯƠNG SAI (MANUAL BP) ---")
print(f"{'R-squared của mô hình phụ':<30}: {r2_bp:.4f}")
print(f"{'Lagrange multiplier statistic':<30}: {lm_statistic:.4f}")

if r2_bp > 0.1:
    print("\nKết luận: R-squared của mô hình phụ cao.")
    print("=> Có dấu hiệu hiện tượng phương sai thay đổi.")
else:
    print("\nKết luận: R-squared của mô hình phụ thấp.")
    print("=> Chưa có bằng chứng rõ rệt về vi phạm giả thiết phương sai không đổi.")

##### C.1.d Xử lý Heteroscedasticity bằng Weighted Least Squares (WLS)

Nếu p-value của test Breusch–Pagan < 0.05, ta bác bỏ giả thiết phương sai không đổi. Khi đó, WLS sẽ gán trọng số thấp cho các điểm có phương sai lớn để cải thiện mô hình.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

residuals = (np.array(y_val).flatten() - y_val_pred_gd.flatten())
residuals_sq = residuals**2

# Hồi quy phụ: residuals^2 ~ X_val_sm
bp_model = LinearRegression()
bp_model.fit(X_val_sm, residuals_sq)
r2_bp = r2_score(residuals_sq, bp_model.predict(X_val_sm))

# LM statistic = n * R2. Giả định ngưỡng bác bỏ (p-value < 0.05) 
is_heteroscedastic = r2_bp > 0.1  # Ngưỡng giả định để kích hoạt WLS

# --- XỬ LÝ WLS ---
if is_heteroscedastic:
    print(f"Phát hiện Heteroscedasticity (R2 phụ: {r2_bp:.4f})! Đang triển khai WLS...")
    
    # 1. Ước lượng trọng số (Weights)
    # 1 / |residual|
    # Thêm 1e-5 để tránh chia cho 0
    weights = 1.0 / (np.abs(residuals) + 1e-5)
    
    # 2. Huấn luyện mô hình WLS bằng Scikit-learn
    wls_model = LinearRegression()
    wls_model.fit(X_val_sm, y_val, sample_weight=weights)
    
    # 3. Dự báo và đánh giá
    y_val_pred_wls = wls_model.predict(X_val_sm)
    rmse_wls = np.sqrt(mean_squared_error(y_val, y_val_pred_wls))
    r2_wls = r2_score(y_val, y_val_pred_wls)

    # Hiển thị kết quả
    print("-" * 30)
    print(f"{'Chỉ số':<20} | {'Giá trị':<10}")
    print("-" * 30)
    print(f"{'RMSE WLS':<20} | {rmse_wls:,.2f}")
    print(f"{'R-squared WLS':<20} | {r2_wls:.4f}")
    print(f"{'Intercept':<20} | {wls_model.intercept_[0] if hasattr(wls_model.intercept_, '__len__') else wls_model.intercept_:.4f}")
    
    # In một vài hệ số quan trọng
    print("\nTop các hệ số (Coefficients):")
    for i, coef in enumerate(wls_model.coef_.flatten()[:5]): # 5 cái đầu
        print(f" Biến {i}: {coef:.4f}")
        
else:
    print("Không phát hiện Heteroscedasticity nghiêm trọng. Có thể giữ nguyên mô hình OLS.")

<a id='c2'></a>

### C.2 Regularization & Feature Selection

**Nội dung:**
- **a) Ridge (L2) và Lasso (L1)**: So sánh 2 regularization phổ biến
- **b) Chọn λ tối ưu bằng k-Fold CV (k=10)**: Dùng grid search và warm start (đối với lasso)
- **c) Regularization Path**: Hệ số thay đổi theo λ (Ridge/Lasso), đánh dấu best λ
- **d) Elastic Net**: Kết hợp L1 + L2, tìm vùng tối ưu (λ₁, λ₂)
- **e) Chọn đặc trưng**: So sánh Forward, Backward, Lasso-based

##### C.2.a Ridge (L2) vs Lasso (L1)

Fit Ridge/Lasso với alpha đơn giản và so sánh RMSE

In [ ]:
from train_utils import *
from sklearn.linear_model import Ridge, Lasso

# Fit ridge và lasso với alpha mặc định
ridge = Ridge(alpha=1.0).fit(X_train_s, y_train)
lasso = Lasso(alpha=0.1).fit(X_train_s, y_train)

print(f"Ridge RMSE: {np.sqrt(mean_squared_error(y_val, ridge.predict(X_val_s))):,.3f}")
print(f"Lasso RMSE: {np.sqrt(mean_squared_error(y_val, lasso.predict(X_val_s))):,.3f}")

##### C.2.b Chọn λ tối ưu (K-Fold CV + Grid Search)

- **Ridge**: Grid Search + K-Fold CV (k=10)
- **Lasso**: LassoCV (built-in warm-start, nhanh hơn)

In [ ]:
# Chọn alpha (50 giá trị từ 10^-4 đến 10^4 trên log scale)
alphas = np.logspace(-4, 4, 50)

# Ridge: Grid Search + K-Fold CV
best_α_r, best_rmse_r, ridge_results = select_best_lambda_cv(X_train_s, y_train, Ridge, alphas, k=10)

# Lasso: LassoCV (warm-start)
best_α_l, lasso_results, _ = select_best_lambda_lasso_path(X_train_s, y_train, alphas, k=10)
best_rmse_l = min(r[1] for r in lasso_results)

print(f"Ridge: λ={best_α_r:.4f} | CV RMSE={best_rmse_r:,.3f}")
print(f"Lasso: λ={best_α_l:.4f} | CV RMSE={best_rmse_l:,.3f}")

**Nhận xét:**

- Đối với ridge, mô hình chọn regularization vừa phải (λ = 36).
- Đối với lasso, mô hình chọn regularization mạnh hơn (λ = 233), gấp khoảng 6 lần so với ridge, do L1 penalty yếu hơn L2 ở cùng λ.

Chênh lệch RMSE không đáng kể (chỉ 0.06%), nhưng ridge xử lí tốt hơn do có các feature tương quan nhau (multicollinearity).

##### C.2.c Regularization Path, hệ số theo log(λ)

- **Ridge**: Hệ số giảm dần khi λ tăng, nhưng không bao giờ về 0
- **Lasso**: Hệ số set về 0 khi λ đủ lớn nên có thể lọc feature
- Đánh dấu best λ bằng red dashed line
- Trục x ngược (trái = regularization mạnh)

In [ ]:
# Regularization Path: Ridge & Lasso
alphas_path = np.logspace(-2, 3, 50)

plot_regularization_path(X_train_s, y_train, alphas_path, feature_names, 
                         model_class=Ridge, best_alpha=best_α_r)
plot_regularization_path(X_train_s, y_train, alphas_path, feature_names, 
                         model_class=Lasso, best_alpha=best_α_l)

##### C.2.d Elastic Net: Kết hợp L1 + L2 (10 fold CV)

$$E(\mathbf{w}) = \frac{1}{2}\|\mathbf{t} - \mathbf{\Phi}\mathbf{w}\|^2 + \lambda_1 \|\mathbf{w}\|_1 + \frac{\lambda_2}{2}\|\mathbf{w}\|^2$$

Grid search trên (λ, l1_ratio): tìm vùng tối ưu, vẽ heatmap RMSE

In [ ]:
# Elastic Net: Grid search (λ, l1_ratio)
alphas_en = np.logspace(-3, 2, 5)
l1_ratios = [0.7, 0.8, 0.9]

rmse_matrix, best_en, results_en = plot_elastic_net_heatmap(
    X_train_s, y_train, alphas_en, l1_ratios, k=3
)

print(f"Best Elastic Net Configuration:")
print(f"λ               = {best_en[0]:.4f}")
print(f"l1_ratio        = {best_en[1]:.4f}")
print(f"Validation RMSE = {best_en[2]:,.3f}")

**Phân tích Vùng Tối ưu (λ, l1_ratio):**

**Heatmap diễn giải:**
- **Trục ngang (l1_ratio):** Tỷ lệ L1 penalty
  - Ở lần coarse search trước, lựa chọn khoảng l1_ratio = [0.2, 0.5, 0.8] cho thấy vùng tối tập trung quanh r1_ratio = 0.8. Do đó, lần fine search này được giới hạn trong khoảng [0.7, 0.8, 0.9] để tiết kiệm chi phí tính toán mà vẫn bảo toàn độ chính xác.

- **Trục dọc (λ):** Độ mạnh của regularization
  - Chọn khoảng λ thấp hơn ridge và lasso (10^-3 ~ 10^2 thay vì 10^-2 ~ 10^3) vì elastic net kết hợp cả L1 và L2 nên penalty được tăng bởi cả hai cùng lúc.

- **Màu sắc (RMSE):** Tím đậm = RMSE tốt, màu sáng = RMSE xấu

**Kết luận:**
Từ λ và l1_ratio, chúng ta phân tích được vùng tối ưu (λ1,λ2):
$$
\begin{aligned}
\lambda_2 &= \lambda \cdot \text{l1\_ratio} = 0.0178 \cdot 0.8 = 0.01424 \\
\lambda_1 &= \lambda \cdot \frac{1 - \text{l1\_ratio}}{2} = 0.0178 \cdot \frac{0.2}{2} = 0.00178 \\
(\lambda_1, \lambda_2) &= (0.00178, 0.01424)
\end{aligned}
$$

##### C.2.e Feature Selection: So sánh 3 phương pháp

1. **Forward Stepwise Selection**: Thêm features dần (greedy)
2. **Backward Elimination**: Loại features dần (greedy)
3. **Lasso-based**: Dựa trên coefficients ≠ 0

So sánh: số features selected + validation RMSE

In [ ]:
# Feature Selection: 3 phương pháp
fwd_idx, fwd_names, fwd_scores = forward_stepwise_selection(X_train_s, y_train, feature_names)
bwd_idx, bwd_names, bwd_scores = backward_elimination(X_train_s, y_train, feature_names)
lasso_idx, lasso_names, lasso_coefs = lasso_feature_selection(X_train_s, y_train, best_α_l, feature_names)

# Tính RMSE cho Lasso
lasso_model_final = Lasso(alpha=best_α_l, max_iter=10000)
lasso_model_final.fit(X_train_s, y_train)
lasso_rmse = np.sqrt(mean_squared_error(y_val, lasso_model_final.predict(X_val_s)))

# In kết quả chi tiết
print(f"1. Forward Stepwise Selection")
print(f"   - Features selected: {len(fwd_names)}")
print(f"   - Validation RMSE: {fwd_scores[-1]:,.3f}")

print(f"2. Backward Elimination")
print(f"   - Features selected: {len(bwd_names)}")
print(f"   - Validation RMSE: {bwd_scores[-1]:,.3f}")

print(f"3. Lasso-based (λ={best_α_l:.4f})")
print(f"   - Features selected: {len(lasso_names)}")
print(f"   - Validation RMSE: {lasso_rmse:,.3f}")


plot_feature_selection_comparison(
    fwd_names, fwd_scores,
    bwd_names, bwd_scores,
    lasso_names, lasso_rmse
)

**So sánh 3 phương pháp Feature Selection:**

| Tiêu chí | Forward Stepwise | Backward Elimination | Lasso-based |
|----------|---|---|---|
| **Cơ chế** | Thêm features dần (Greedy) | Loại features dần (Greedy) | Shrink & zero-out (L1) |
| **Ưu điểm** | - Nhanh<br>- Giữ features quan trọng | - Xử lý multicollinearity<br>- Thuyết phục | - Tự động (không greedy)<br>- Dùng ML model |
| **Nhược điểm** | - Greedy (local optimum)<br>- Không xử lý multicollinearity | - Greedy (local optimum)<br>- Loại quá nhiều features | - Có thể loại features tốt<br>- Phụ thuộc vào λ |
| **Số features** | Thường nhiều | Thường ít nhất | Trung bình |
| **Validation RMSE** | Phụ thuộc dữ liệu | Phụ thuộc dữ liệu | Phụ thuộc λ |

**Nhận xét thực hành:**

- Forward Stepwise có RSME thấp nhất (68,458), hiệu quả nhất trong 3 phương pháp.
- Backward Elimination có RMSE cao nhất (83,644) - kết quả tệ nhất, không đáng tin cậy.
- Lasso-based có RMSE cao hơn forward một chút (70,603) - nhưng model đơn giản hơn (14 vs 16 features).<br>
**Do đó:** Nên dùng Forward Stepwise để có RSME thấp nhất, hoặc dùng Lasso-based để cân bằng giữa accuracy và simplicity.



<a id='c3'></a>

### C.3 Mô hình với Hàm Cơ sở Phi Tuyến và Ablation Study

**Nội dung**

- **a) Áp dụng hàm cơ sở phi tuyến**, ít nhất ba loại hàm cơ sở (polynomial, Gaussian RBF, và một loại
tự chọn).
- **b) Vẽ validation curve:** MSE theo bậc đa thức / số hàm cơ sở.
- **c) Thực hiện ablation study:** lần lượt bỏ từng nhóm đặc trưng hoặc từng loại
hàm cơ sở, đo tác động đến hiệu năng – từ đó xác định đặc trưng / hàm cơ sở
quan trọng nhất.
- **d) Phân tích hiệu ứng tương tác:** thêm các hạng xixj vào mô hình và đánh giá
mức cải thiện

##### C.3.a Áp dụng hàm cơ sở phi tuyến (Non-linear Basis Functions) 

**Ba loại hàm cơ sở:**
1. **Polynomial Features**: Bắt quan hệ phi tuyến bậc cao (e.g., `x₁², x₁x₂, x₂²`)
2. **Gaussian RBF**: Đo khoảng cách từ mỗi điểm đến các center - phù hợp với dữ liệu địa lý (longitude/latitude)
3. **Spline (Cubic Splines)**: Chia miền giá trị thành các đoạn và áp dụng cubic polynomial trên mỗi đoạn
   - **Knots**: Điểm chia ranh giới giữa các đoạn (tự động phân phối đều)
   - **Degree**: Bậc polynomial (3 = cubic)
   - Ưu điểm: Smooth, linh hoạt, không cần tuning parameter phức tạp như RBF/Sigmoid
   - Tự động thích nghi với data distribution

In [ ]:
# Import basis function utilities
from train_utils import (
    SplineBasis,
    apply_basis_function, 
    plot_validation_curve_polynomial, 
    plot_validation_curve_rbf,
    plot_validation_curve_spline
)

In [ ]:
# 1. Polynomial Features
print("=" * 45)
print("1. POLYNOMIAL (#3)")
print("=" * 45)

# Apply polynomial
X_train_poly, X_val_poly, X_test_poly, poly_transformer = apply_basis_function(
    X_train_s, X_val_s, X_test_s,
    basis_type='polynomial',
    degree=2
)

print(f"Original shape: {X_train_s.shape}")
print(f"After polynomial (degree=2): {X_train_poly.shape}")
# print(f"New dimensions: {poly_transformer.get_feature_names_out(feature_names)[:10]}...")  # Show first 10

# Ridge fit
ridge_poly = Ridge(alpha=1.0)
ridge_poly.fit(X_train_poly, y_train)
poly_val_rmse = np.sqrt(mean_squared_error(y_val, ridge_poly.predict(X_val_poly)))
print(f"Validation RMSE: {poly_val_rmse:,.0f}\n")

In [ ]:
# 2. Gaussian RBF
print("=" * 45)
print("2. GAUSSIAN RBF (#2)")
print("=" * 45)

# Tính khoảng cách trung bình giữa các điểm để chọn gamma
from sklearn.metrics.pairwise import euclidean_distances
distances = euclidean_distances(X_train_s[:1000])
avg_distance = np.mean(distances[distances > 0])
chosen_gamma = 1 / (2 * avg_distance**2)
print(f"Gamma = {chosen_gamma:.6f}")

# Apply RBF
X_train_rbf, X_val_rbf, X_test_rbf, rbf_transformer = apply_basis_function(
    X_train_s, X_val_s, X_test_s,
    basis_type='rbf',
    n_centers=20,
    gamma=chosen_gamma
)

print(f"Original shape: {X_train_s.shape}")
print(f"After RBF (n_centers=20, random sampling): {X_train_rbf.shape}")

# Ridge fit
ridge_rbf = Ridge(alpha=1.0)
ridge_rbf.fit(X_train_rbf, y_train)
rbf_val_rmse = np.sqrt(mean_squared_error(y_val, ridge_rbf.predict(X_val_rbf)))
print(f"Validation RMSE: {rbf_val_rmse:,.0f}\n")

In [ ]:
# 3. Spline Basis
print("=" * 45)
print("3. SPLINE (#3)")
print("=" * 45)

# Apply Spline
X_train_spline, X_val_spline, X_test_spline, spline_transformer = apply_basis_function(
    X_train_s, X_val_s, X_test_s,
    basis_type='spline',
    n_knots=5,
    degree=3
)

print(f"Original shape: {X_train_s.shape}")
print(f"After spline (n_knots=5, degree=3): {X_train_spline.shape}")

# Ridge fit
ridge_spline = Ridge(alpha=1.0)
ridge_spline.fit(X_train_spline, y_train)
spline_val_rmse = np.sqrt(mean_squared_error(y_val, ridge_spline.predict(X_val_spline)))
print(f"Validation RMSE: {spline_val_rmse:,.0f}\n")

**Tổng kết:**

| Basis function | Số features sau áp dụng | Validation RMSE |
|---|---|---|
| **Polynomial** | 152 | 68,385 |
| **RBF** | 20 | 72,529 |
| **Spline** | 96 | 64,504 |

##### C.3.b Vẽ validation curve MSE theo bậc đa thức / số hàm cơ sở

Vẽ validation curves cho 3 loại basis functions:
- RMSE theo bậc đa thức (Polynomial)
- RMSE theo số RBF centers (RBF)  
- RMSE theo số Spline knots (Spline Basis)

In [ ]:
print("=" * 45)
print("VALIDATION CURVE: RMSE theo bậc đa thức")
print("=" * 45)

# Compute validation curve for polynomial degrees
poly_results = plot_validation_curve_polynomial(
    X_train_s, y_train, X_val_s, y_val,
    degrees=[1, 2, 3, 4],
    model_class=Ridge,
    alpha=1.0
)

print("\nResults Table:")
print(poly_results.to_string(index=False))

**Nhận xét:**
- Degree 1: linear model (underfitting) - RMSE (train lẫn val) vẫn còn lớn
- Degree 2: optimal zone - RMSE val đạt giá trị nhỏ nhất
- Degree 3+: RMSE val tăng mạnh dẫn tới overfitting gap cũng tăng mạnh theo

In [ ]:
print("\n" + "=" * 45)
print("VALIDATION CURVE: RMSE theo số hàm cơ sở")
print("=" * 45)

# Validation curve cho RBF theo số centers
rbf_results = plot_validation_curve_rbf(
    X_train_s, y_train, X_val_s, y_val,
    n_centers_list=[10, 25, 50, 100, 150, 200, 300],
    gamma=chosen_gamma,
    model_class=Ridge,
    alpha=1.0
)

print("\nResults Table:")
print(rbf_results.to_string(index=False))


**Nhận xét:**
- Với số center tăng dần từ 10 - 500, RMSE của cả train lẫn val đều giảm ổn định và không có dấu hiệu overfitting. Trong range trên, số center = 500 là tốt nhất.
- Tăng thêm số center khả năng cao sẽ giúp mô hình giảm RMSE nhưng cũng tăng mạnh chi phí compile.


In [ ]:
print("\n" + "=" * 45)
print("VALIDATION CURVE: RMSE theo số Spline knots")
print("=" * 45)

# Compute validation curve for Spline basis (vary n_knots)
spline_results = plot_validation_curve_spline(
    X_train_s, y_train, X_val_s, y_val,
    n_knots_list=[3, 4, 7, 10, 15, 30, 50],
    degree=3,
    model_class=Ridge,
    alpha=1.0
)

print("\nResults Table:")
print(spline_results.to_string(index=False))

**Nhận xét:**
- Số knot từ 3 - 7: RMSE cả train lẫn val giảm tốt, gap đạt thấp nhất ở số knot = 7
- Số knot từ 10 - 15: RMSE cả train lẫn val vẫn giảm nhưng gap có dấu hiệu tăng
- Số knot từ 30 - 50: đầu giai đoạn overfitting, gap tăng rõ rệt


<a id='c3c'></a>

##### C.3.c Thực hiện Ablation Study (Đánh giá tầm quan trọng của đặc trưng)

**Mục tiêu:** Lần lượt loại bỏ từng đặc trưng (hoặc nhóm đặc trưng) khỏi tập dữ liệu, huấn luyện lại mô hình (Ridge Regression) và đo lường sự thay đổi của Validation RMSE.
- Nếu RMSE **tăng mạnh** khi bỏ một đặc trưng $\rightarrow$ Đặc trưng đó **rất quan trọng**.
- Nếu RMSE **không đổi hoặc giảm** $\rightarrow$ Đặc trưng đó **ít quan trọng hoặc gây nhiễu**.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error

# 1. Huấn luyện mô hình Baseline (sử dụng tất cả features)
baseline_model = Ridge(alpha=1.0)
baseline_model.fit(X_train_s, y_train)
y_val_pred_base = baseline_model.predict(X_val_s)
base_rmse = np.sqrt(mean_squared_error(y_val, y_val_pred_base))

print(f"Baseline Validation RMSE: {base_rmse:,.0f}\n")

# 2. Thực hiện Ablation Study
ablation_results = {}

for i, feature in enumerate(feature_names):
    # Loại bỏ feature thứ i (cột i)
    X_train_ablated = np.delete(X_train_s, i, axis=1)
    X_val_ablated = np.delete(X_val_s, i, axis=1)
    
    # Huấn luyện và dự đoán
    model = Ridge(alpha=1.0)
    model.fit(X_train_ablated, y_train)
    y_pred_ablated = model.predict(X_val_ablated)
    
    # Tính RMSE và độ lệch so với baseline
    ablated_rmse = np.sqrt(mean_squared_error(y_val, y_pred_ablated))
    rmse_increase = ablated_rmse - base_rmse
    ablation_results[feature] = rmse_increase

# 3. Trực quan hóa kết quả
ablation_df = pd.DataFrame(list(ablation_results.items()), columns=['Feature', 'RMSE_Increase'])
ablation_df = ablation_df.sort_values(by='RMSE_Increase', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x='RMSE_Increase', y='Feature', data=ablation_df, palette='viridis')
plt.axvline(x=0, color='red', linestyle='--')
plt.title("Ablation Study: Tác động khi loại bỏ từng đặc trưng")
plt.xlabel("Mức tăng RMSE")
plt.ylabel("Đặc trưng bị loại bỏ")
plt.tight_layout()
plt.show()

**Nhận xét (Ablation Study):**
- Biến **`median_income`** chắc chắn là biến quan trọng nhất (như đã thấy ở phần EDA). Khi loại bỏ biến này, RMSE tăng vọt, chứng tỏ mô hình mất đi lượng thông tin dự báo lớn nhất.
- Vị trí địa lý (`longitude`, `latitude`) cũng đóng vai trò cốt lõi. Bỏ một trong hai sẽ làm giảm hiệu năng đáng kể.
- Một số biến có mức tăng RMSE gần 0 hoặc âm (nằm bên trái đường đỏ) có thể được xem xét loại bỏ hoàn toàn khỏi mô hình để giảm độ phức tạp mà không làm mất độ chính xác.

<a id='c3d'></a>

##### C.3.d Phân tích Hiệu ứng Tương tác (Interaction Effects)

**Mục tiêu:** Khám phá xem liệu sự kết hợp của 2 đặc trưng (hạng tử $x_i x_j$) có mang lại thông tin mới cho mô hình hay không.
Ta sử dụng `PolynomialFeatures(interaction_only=True)` để sinh ra tất cả các cặp tương tác có thể có, sau đó so sánh hiệu năng với mô hình gốc.

In [ ]:
from sklearn.preprocessing import PolynomialFeatures

# 1. Tạo các hạng tử tương tác (chỉ xi * xj, không chứa xi^2)
poly_interact = PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)
X_train_interact = poly_interact.fit_transform(X_train_s)
X_val_interact = poly_interact.transform(X_val_s)

# Lấy tên các đặc trưng mới
interact_feature_names = poly_interact.get_feature_names_out(feature_names)

print(f"Số lượng features gốc: {X_train_s.shape[1]}")
print(f"Số lượng features sau khi thêm tương tác: {X_train_interact.shape[1]}")

# 2. Huấn luyện mô hình với tập dữ liệu chứa biến tương tác
ridge_interact = Ridge(alpha=100.0)
ridge_interact.fit(X_train_interact, y_train)

# 3. Đánh giá và so sánh
y_val_pred_interact = ridge_interact.predict(X_val_interact)
rmse_interact = np.sqrt(mean_squared_error(y_val, y_val_pred_interact))

print(f"\nValidation RMSE (Baseline): {base_rmse:,.0f}")
print(f"Validation RMSE (Interaction): {rmse_interact:,.0f}")
improvement = base_rmse - rmse_interact
print(f"Cải thiện: {improvement:,.0f} ({improvement/base_rmse*100:.2f}%)")

# 4. Trực quan hóa 10 hạng tử tương tác có ảnh hưởng lớn nhất
# Sử dụng giá trị tuyệt đối của hệ số để đánh giá tầm quan trọng
coef_df = pd.DataFrame({
    'Feature': interact_feature_names,
    'Coefficient': ridge_interact.coef_.flatten(),
    'Abs_Coefficient': np.abs(ridge_interact.coef_.flatten())
})
# Chỉ lọc ra các biến là biến tương tác (có dấu khoảng trắng do PolynomialFeatures)
interact_only_df = coef_df[coef_df['Feature'].str.contains(' ')].sort_values(by='Abs_Coefficient', ascending=False).head(10)

plt.figure(figsize=(10, 6))
sns.barplot(x='Coefficient', y='Feature', data=interact_only_df, palette='coolwarm')
plt.title("Top 10 Đặc trưng Tương tác quan trọng nhất (Dựa trên Hệ số Ridge)")
plt.xlabel("Giá trị Hệ số (Coefficient)")
plt.ylabel("Hạng tử Tương tác ($x_i x_j$)")
plt.tight_layout()
plt.show()

Phân tích Hiệu ứng Tương tác:

Hiệu ứng Vùng vịnh: Tương tác giữa tọa độ và vị trí gần vịnh (NEAR BAY) cho thấy sự sụt giảm giá trị nhà rất nhanh khi rời xa các điểm nóng địa lý này.

Mật độ dân cư: Sự kết hợp giữa quy mô hộ gia đình và tổng số phòng/phòng ngủ cho thấy mô hình đã bắt được đặc điểm của các khu dân cư cao cấp (rộng rãi, ít người) đối lập với các khu vực bình dân (chật hẹp, đông người).

Kết luận: Việc thêm các hạng tử tương tác giúp mô hình "thông minh" hơn khi hiểu rằng giá trị của một đặc trưng phụ thuộc rất nhiều vào bối cảnh của các đặc trưng khác.